# 03. Бизнес-выводы и продуктовые рекомендации

В этом ноутбуке результаты статистического анализа из ноутбука 02 переводятся в три конкретные продуктовые рекомендации. Каждая рекомендация формулируется в виде гипотезы, привязанной к измеримой метрике и допускающей проверку через A/B-тест.

Сводный тезис: окно эффективного воздействия на retention составляет первые тридцать дней; основная целевая аудитория для триггерных кампаний это клиенты с первым чеком ниже медианы; декабрьские когорты требуют отдельной стратегии работы.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

customers = pd.read_parquet('../data/customers_labeled.parquet')
retention = pd.read_parquet('../data/cohort_retention.parquet')
print(f'Клиентов в сегментации: {len(customers):,}')

## Сводка результатов анализа

| Наблюдение | Количественная характеристика | Бизнес-значение |
|---|---|---|
| Падение retention сосредоточено в первом месяце | Со 100% до примерно 22% | Окно воздействия: первые 30 дней |
| Первый чек связан с вероятностью возврата | t-критерий Уэлча, p < 0.001 | Возможность раннего скоринга клиентов |
| Декабрьские когорты показывают сниженный retention | Видно на тепловой карте | Необходима отдельная стратегия по сегменту |

## Рекомендация 1. Таргетированная триггерная кампания реактивации

Содержание. Запуск персональной email- или push-кампании с предложением второй покупки, ограниченной сегментом клиентов с первым чеком ниже медианы. Время отправки: первые 14 дней после первой транзакции.

Обоснование. Сегмент с высоким первым чеком уже без вмешательства обеспечивает достаточный уровень возврата, и дополнительные стимулы здесь приводят преимущественно к каннибализации маржи. Сегмент с низким первым чеком располагает наибольшим неосвоенным потенциалом прироста retention. Окно 14 дней выбрано исходя из формы кривой удержания: после первого месяца возможности воздействия резко сокращаются.

In [ ]:
low_check = customers[customers['check_segment'].str.startswith('Низкий')]
high_check = customers[customers['check_segment'].str.startswith('Высокий')]

lift_potential = high_check['returned'].mean() - low_check['returned'].mean()

print(f'Размер целевого сегмента (низкий чек):  {len(low_check):,} клиентов')
print(f'Текущая доля возврата в сегменте:        {low_check["returned"].mean()*100:.1f}%')
print(f'Доля возврата в сегменте "высокий чек":  {high_check["returned"].mean()*100:.1f}%')
print(f'Разрыв (потенциал апсайда):              {lift_potential*100:.1f} п.п.')

Дизайн измерения. Целевой сегмент случайным образом делится на две равные группы; экспериментальная получает триггерное письмо, контрольная получает стандартный пользовательский путь. Основная метрика: доля клиентов, совершивших вторую покупку в течение 30 дней. Контрольные (guardrail) метрики: средний размер второй покупки и совокупная выручка с клиента в течение 60 дней. Минимально детектируемый эффект задаётся равным 2 процентным пунктам в абсолюте. Расчёт необходимого размера выборки и валидация дизайна через симуляцию выполнены в ноутбуке 05.

## Рекомендация 2. Отдельная коммуникация для декабрьских когорт

Содержание. Клиенты с первой покупкой в декабре исключаются из стандартного потока январских и февральских реактивационных кампаний. Вместо скидочных стимулов им направляется приветственная серия с контентом о категориях, не представленных в первой покупке.

Обоснование. Декабрьский спрос содержит значительную долю покупок-подарков, мотив которых не соответствует профилю целевой аудитории бренда. Стандартная кампания с промокодом на повторение прошлой покупки в этом контексте малоэффективна и расходует маркетинговый бюджет с низкой отдачей.

In [ ]:
december_mask = retention.index.month == 12
dec_retention_m1 = retention.loc[december_mask, 1].mean()
other_retention_m1 = retention.loc[~december_mask, 1].mean()

print(f'Retention M+1 декабрьских когорт:  {dec_retention_m1:.1f}%')
print(f'Retention M+1 остальных когорт:    {other_retention_m1:.1f}%')
print(f'Разница:                            {other_retention_m1 - dec_retention_m1:.1f} п.п.')

## Рекомендация 3. Внедрение признака размера первого чека в скоринговый контур

Содержание. Признак «первый чек выше или ниже медианы» добавляется в продуктовую аналитику в качестве раннего предиктора LTV. Используется в дашбордах онбординга и в моделях прогноза ценности клиента.

Обоснование. Признак доступен немедленно после первой транзакции и не требует накопления истории. Это позволяет CRM-команде применять минимальную сегментацию ещё до получения других сигналов о клиенте.

Ограничения применения. Сегментация по первому чеку обоснована для решений, направленных на пользователя извне (реактивационные кампании, ретаргетинг). Использовать её для дискриминации внутри пользовательского пути (например, ограничивать ассортимент) недопустимо: это приводит к самоисполняющемуся прогнозу и снижает совокупную выручку.

## Дополнительный анализ

Для подкрепления рекомендаций к проекту приложены три ноутбука с практическими расчётами:

- 04. LTV и unit-экономика. Расчёт наблюдаемого LTV по сегментам, проверка концентрации выручки по правилу Парето, моделирование чувствительности окупаемости к стоимости привлечения.
- 05. Дизайн A/B-теста. Расчёт необходимого размера выборки для MDE = 2 п.п., построение power curve, валидация дизайна через Монте-Карло симуляцию с проверкой эмпирической мощности и доли ложноположительных результатов.
- 06. RFM-сегментация. Разбиение клиентской базы по трём осям (Recency, Frequency, Monetary), формирование интерпретируемых сегментов и плана работы по каждому из них.

## Ограничения проекта

1. Датасет не содержит информации об источнике трафика и канале привлечения. На реальном маркетплейсе retention существенно зависит от канала, и без этого среза анализ остаётся неполным.
2. Существенная часть выручки приходится на оптовых клиентов. Их поведение принципиально отличается от B2C-сегмента, и в идеале они должны анализироваться отдельно.
3. Предиктивные модели возврата сознательно не строились. Поставленная бизнес-задача решается методами описательной статистики при сохранении интерпретируемости; ML добавил бы предсказательную точность, но не качество выводов.
4. Связь между размером первого чека и возвратом установлена корреляционно. Доказательство причинной природы эффекта требует контролируемого эксперимента, дизайн которого представлен в ноутбуке 05.

## Итоговая формулировка

Эффективная стратегия удержания строится на разделении базы по потенциалу прироста, а не на однородном воздействии на всех клиентов. Размер первого чека предоставляет ранний сигнал, на основе которого можно принимать обоснованные решения о распределении retention-бюджета.